# Creating tables and constraints

Practice notebook for the datafromzero SQL lesson **Creating tables and constraints**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

Every table you create here is new. The cafe-sales tables are never changed, and the setup cell gives you a clean start.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it. Run it again at any time to start over.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Create `menu` with its rules

In [ ]:
con.sql("""
CREATE TABLE menu (
  product_id INTEGER PRIMARY KEY,
  product_name VARCHAR NOT NULL UNIQUE,
  category VARCHAR NOT NULL CHECK (category IN ('coffee', 'tea', 'bakery', 'food')),
  unit_price_usd DECIMAL(6, 2) NOT NULL CHECK (unit_price_usd > 0),
  is_active BOOLEAN DEFAULT true
)
""").show()

### Step 2. Fill it from `products`, then read the rules back

In [ ]:
con.sql("""
INSERT INTO menu (product_id, product_name, category, unit_price_usd)
SELECT * FROM products
""").show()

In [ ]:
con.sql("""
DESCRIBE menu
""").show()

### Step 3. Create `stock`, with a two-column key and a foreign key

In [ ]:
con.sql("""
CREATE TABLE stock (
  store_id INTEGER NOT NULL CHECK (store_id IN (1, 2, 3)),
  product_id INTEGER NOT NULL REFERENCES menu (product_id),
  units_on_hand INTEGER NOT NULL DEFAULT 0 CHECK (units_on_hand >= 0),
  PRIMARY KEY (store_id, product_id)
)
""").show()

### Step 4. Add rows that obey the rules

In [ ]:
con.sql("""
INSERT INTO stock VALUES (1, 1, 40), (1, 10, 25), (1, 6, 12);
INSERT INTO stock (store_id, product_id) VALUES (1, 7);
SELECT * FROM stock ORDER BY product_id
""").show()

### Step 5. Try four rows that break the rules

Each of the next four cells prints an error instead of a table. That is the point: read the message, then go to the next cell.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO stock VALUES (1, 99, 10)
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO stock VALUES (2, 1, -5)
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO stock VALUES (1, 1, 20)
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO stock VALUES (NULL, 2, 20)
    """).show()
except Exception as e:
    print(e)


### Step 6. Verify

In [ ]:
con.sql("""
SELECT COUNT(*) AS stock_rows FROM stock
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS items, SUM(unit_price_usd) AS total FROM menu
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS items, SUM(unit_price_usd) AS total FROM products
""").show()

## Exercise 1. A table of loyalty perks that accepts only the three real tiers

The `customers` table uses three tiers: `bronze`, `silver` and `gold`. Finish the `CREATE TABLE`: make `loyalty_tier` the primary key and accept only the three tiers; let `discount_pct` default to 0 and accept only 0 to 50. The first line drops an earlier try, so you can run the cell again. Then add a cell that inserts `'bronze'` with the perk `'free refill'` and no discount, and a cell that selects the table.

In [ ]:
con.sql("""
DROP TABLE IF EXISTS tier_perk;
CREATE TABLE tier_perk (
  loyalty_tier VARCHAR,   -- add the primary key and the CHECK here
  perk VARCHAR NOT NULL,
  discount_pct INTEGER    -- add the DEFAULT and the CHECK here
)
""").show()

In [ ]:
# Insert bronze with the perk free refill and no discount

In [ ]:
# Select the table

## Exercise 2. Which of six new menu items will `menu_ex2` accept?

Done for you: the next cell builds `menu_ex2`, with the same rules as `menu` and the 16 items of `products`, and `new_items`, with the six candidate rows. Run it, then look at `new_items`.

In [ ]:
con.sql("""
CREATE TABLE menu_ex2 (
  product_id INTEGER PRIMARY KEY,
  product_name VARCHAR NOT NULL UNIQUE,
  category VARCHAR NOT NULL CHECK (category IN ('coffee', 'tea', 'bakery', 'food')),
  unit_price_usd DECIMAL(6, 2) NOT NULL CHECK (unit_price_usd > 0),
  is_active BOOLEAN DEFAULT true
);
INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd) SELECT * FROM products;
CREATE TABLE new_items AS
SELECT * FROM (VALUES
  (17, 'Oat Latte', 'coffee', 5.25),
  (18, 'Espresso', 'coffee', 3.00),
  (19, 'Brownie', 'dessert', 3.50),
  (20, 'Free Water', 'food', 0.00),
  (1, 'Cortado', 'coffee', 4.00),
  (21, NULL, 'tea', 3.00)
) AS t (product_id, product_name, category, unit_price_usd);
SELECT * FROM new_items
""").show()

Before you run the inserts, write one line per candidate here: accepted or refused, and by which rule.

In [ ]:
# Predictions, one line per product_id:
#

Now insert the candidates one at a time. The refused ones print an error: that is expected.

In [ ]:
con.sql("""
INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
SELECT * FROM new_items WHERE product_id = 17
""").show()

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
    SELECT * FROM new_items WHERE product_id = 18
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
    SELECT * FROM new_items WHERE product_id = 19
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
    SELECT * FROM new_items WHERE product_id = 20
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
    SELECT * FROM new_items WHERE product_id = 1
    """).show()
except Exception as e:
    print(e)


In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    INSERT INTO menu_ex2 (product_id, product_name, category, unit_price_usd)
    SELECT * FROM new_items WHERE product_id = 21
    """).show()
except Exception as e:
    print(e)


## Exercise 3. Spot the flaw

Done for you: the next cell builds `menu_ex3`, which is `menu` again, with its rules. A colleague copies it with `CREATE TABLE ... AS SELECT` and says the copy keeps the primary key. Run the colleague's two cells, then find out what went wrong and write the fix.

In [ ]:
con.sql("""
CREATE TABLE menu_ex3 (
  product_id INTEGER PRIMARY KEY,
  product_name VARCHAR NOT NULL UNIQUE,
  category VARCHAR NOT NULL CHECK (category IN ('coffee', 'tea', 'bakery', 'food')),
  unit_price_usd DECIMAL(6, 2) NOT NULL CHECK (unit_price_usd > 0),
  is_active BOOLEAN DEFAULT true
);
INSERT INTO menu_ex3 (product_id, product_name, category, unit_price_usd) SELECT * FROM products;
SELECT COUNT(*) AS items FROM menu_ex3
""").show()

In [ ]:
con.sql("""
CREATE TABLE menu_copy AS SELECT * FROM menu_ex3
""").show()

In [ ]:
con.sql("""
INSERT INTO menu_copy VALUES (1, 'Espresso', 'coffee', 3.00, true)
""").show()

In [ ]:
# What does DESCRIBE menu_copy show?

In [ ]:
# Your fix: create the table with its rules, fill it, and repeat the Espresso insert